In [1]:
import numpy as np
import pandas as pd
import re
from pandas.api.types import (
    is_numeric_dtype,
    is_categorical_dtype,
    is_object_dtype,
)
from scipy.stats import spearmanr

In [29]:
test = pd.read_csv("../feature_data/all_feature_test.csv")
validation = pd.read_csv("../feature_data/all_feature_val.csv")
train = pd.read_csv("../feature_data/all_feature_train.csv")
train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,canceled_count,applications_12m,insured_count,approval_rate,avg_application_amount,avg_credit_application_ratio,latest_credit_amount,latest_annuity,latest_days_decision,days_since_last_application
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,1.0,179055.00,1.000000,179055.0,9251.775,-606.0,606.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,2.0,1.0,435436.50,1.057664,1035882.0,98356.995,-746.0,746.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,1.0,24282.00,0.828021,20106.0,5357.250,-815.0,815.0
3,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,3.0,1.0,150530.25,1.046356,274288.5,16037.640,-374.0,374.0
4,100008,0,Cash loans,M,N,Y,0,99000.0,490495.5,27517.5,...,1.0,1.0,1.0,0.8,194627.25,0.978569,0.0,17885.835,-82.0,82.0


## feature profiling

Mục tiêu không phải để quyết định giữ hay loại biến ngay lập tức mà nhằm xây dựng một bức tranh tổng thể về chất lượng của từng feature trước khi thực hiện Variable Selection.

Toàn bộ quá trình được thực hiện tự động trên tất cả các cột ngoại trừ SK_ID_CURR và TARGET. Đối với mỗi feature, cần thống kê đầy đủ các thông tin như kiểu dữ liệu, số lượng giá trị khác nhau (cardinality), tỷ lệ missing, tỷ lệ giá trị xuất hiện nhiều nhất (dominant value ratio), các thống kê mô tả đối với biến số, cũng như Bad Rate theo từng nhóm giá trị hoặc theo các quantile đối với biến liên tục.

output: một bảng Feature Profile Summary, có thể rà soát hàng trăm feature.

In [3]:
# Feature Profile
def feature_profile(series: pd.Series):

    profile = {}

    # Basic Information
    profile["dtype"] = str(series.dtype)
    profile["missing_pct"] = series.isna().mean()
    profile["n_unique"] = series.nunique(dropna=False)

    # Dominant Value
    value_counts = series.value_counts(dropna=False)

    profile["dominant_pct"] = (
        value_counts.iloc[0] / len(series)
        if len(value_counts) > 0
        else np.nan
    )

    # Numeric Summary
    if is_numeric_dtype(series):

        profile["has_inf"] = np.isinf(series).any()

        finite = series.replace([np.inf, -np.inf], np.nan)

        profile["min"] = finite.min()
        profile["max"] = finite.max()

    else:

        profile["has_inf"] = False
        profile["min"] = np.nan
        profile["max"] = np.nan

    return profile


# Screening Rules
def detect_constant(profile):

    return profile["n_unique"] <= 1


def detect_identifier(feature_name):

    feature_name = feature_name.upper()

    keywords = [
        "SK_ID",
        "ID",
        "INDEX"
    ]

    return any(k in feature_name for k in keywords)


def detect_near_zero_variance(profile, dominant_threshold=0.90):
    return profile["dominant_pct"] >= dominant_threshold


def detect_high_missing(profile, missing_threshold=0.80):

    return profile["missing_pct"] >= missing_threshold


def detect_high_cardinality(series, profile, high_cardinality=50):
    if is_numeric_dtype(series):
        return False

    return profile["n_unique"] > high_cardinality


def detect_inf(profile):

    return profile["has_inf"]

# Decision Engine
def profile_decision(feature_name, series, profile, dominant_threshold, missing_threshold):

    # Constant
    if detect_constant(profile):

        return "Drop", "Constant Feature", "Remove"

    # Identifier
    if detect_identifier(feature_name):

        return "Drop", "Identifier", "Remove"

    # Near Zero Variance
    if detect_near_zero_variance(profile, dominant_threshold):

        return "Drop", "Near Zero Variance", "Remove"

    # Infinite Values
    if detect_inf(profile):

        return "Review", "Contains Inf", "Replace Inf Before Binning"

    # High Missing
    if detect_high_missing(profile, missing_threshold):

        return "Review", "High Missing", "Evaluate Missing Bin"

    # High Cardinality
    if detect_high_cardinality(series, profile):

        return "Review", "High Cardinality", "Consider Category Grouping"

    return "Keep", "Pass", "Fine Classing"


# Main Function
def feature_screening(df, target="TARGET", id="SK_ID_CURR", missing_threshold=0.80, dominant_threshold=0.90):

    features = [col for col in df.columns if col not in [target, id]]

    logs = []

    for feature in features:

        series = df[feature]

        profile = feature_profile(series)

        decision, reason, next_step = profile_decision(
            feature_name=feature,
            series=series,
            profile=profile,
            dominant_threshold=dominant_threshold,
            missing_threshold=missing_threshold
        )

        logs.append({

            "feature": feature,

            **profile,

            "decision": decision,

            "reason": reason,

            "next_step": next_step
        })

    profile_logs = pd.DataFrame(logs)

    keep_features = profile_logs.loc[profile_logs["decision"] != "Drop", "feature"].tolist()

    screened_train = df[[id, target] + keep_features]

    return profile_logs, screened_train

In [4]:
profile_logs, screened_train = feature_screening(train, target = "TARGET", id = "SK_ID_CURR", missing_threshold = 0.80, dominant_threshold = 0.90)


In [5]:
profile_logs

,feature,dtype,missing_pct,n_unique,dominant_pct,has_inf,min,max,decision,reason,next_step
0,NAME_CONTRACT_TYPE,str,0.000000,2,0.904384,False,NaN,NaN,Drop,Near Zero Variance,Remove
1,CODE_GENDER,str,0.000000,3,0.657921,False,NaN,NaN,Keep,Pass,Fine Classing
2,FLAG_OWN_CAR,str,0.000000,2,0.660076,False,NaN,NaN,Keep,Pass,Fine Classing
3,FLAG_OWN_REALTY,str,0.000000,2,0.694110,False,NaN,NaN,Keep,Pass,Fine Classing
4,CNT_CHILDREN,int64,0.000000,12,0.699819,False,0.0,1.900000e+01,Keep,Pass,Fine Classing
...,...,...,...,...,...,...,...,...,...,...,...
211,avg_credit_application_ratio,float64,0.059952,166992,0.069758,False,0.2,2.740852e+00,Keep,Pass,Fine Classing
212,latest_credit_amount,float64,0.053220,42407,0.250858,False,0.0,4.085550e+06,Keep,Pass,Fine Classing
213,latest_annuity,float64,0.054595,109980,0.054595,False,0.0,3.004254e+05,Keep,Pass,Fine Classing
214,latest_days_decision,float64,0.053220,2922,0.053220,False,-2922.0,-1.000000e+00,Keep,Pass,Fine Classing


In [6]:
screened_train.head()

,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,canceled_count,applications_12m,insured_count,approval_rate,avg_application_amount,avg_credit_application_ratio,latest_credit_amount,latest_annuity,latest_days_decision,days_since_last_application
0,100002,1,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,...,0.0,0.0,0.0,1.0,179055.00,1.000000,179055.0,9251.775,-606.0,606.0
1,100003,0,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,...,0.0,0.0,2.0,1.0,435436.50,1.057664,1035882.0,98356.995,-746.0,746.0
2,100004,0,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,...,0.0,0.0,0.0,1.0,24282.00,0.828021,20106.0,5357.250,-815.0,815.0
3,100007,0,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,...,0.0,0.0,3.0,1.0,150530.25,1.046356,274288.5,16037.640,-374.0,374.0
4,100008,0,M,N,Y,0,99000.0,490495.5,27517.5,454500.0,...,1.0,1.0,1.0,0.8,194627.25,0.978569,0.0,17885.835,-82.0,82.0


In [7]:
screened_train.columns.tolist()

['SK_ID_CURR',
 'TARGET',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'OWN_CAR_AGE',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_PHONE',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',
 'LANDAREA_AVG',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVIN

## Fine Classing

Nội dung của bước này là ứng với mỗi feature, ta chia feature đó thành nhiều nhóm (bins) nhỏ để quan sát sự thay đổi của Bad Rate trên toàn
bộ miền giá trị của biến:

- Đối với các biến liên tục, thì ta chia theo quantile nghĩa là mỗi bins sẽ chứa khoảng quan sát xấp xỉ nhau để đảm bảo Bad Rate có thể được quan sát mang tính thống kê. Với các biến phân loại thì giữ nguyên theo từng nhóm như lúc đầu (vì dạng categorical sẽ chỉ có khoảng 5-10 giá trị riêng biệt).  

- Đồng thời với missing ta xây dựng bins riêng, vì có thể nhiều biến thì việc missing cũng mang ý nghĩa tín dụng.

- Report trong đoạn code tương ứng với các chỉ số thống kê trong tùng bins của 1 feature tương ứng


Output là một bảng thống kê ứng với mỗi biến để ta quyết định xem việc merge bins ở các bước tiếp theo.  

Và sau cùng ta sẽ loại những biến có chỉ số IV (infomation Value) thuộc khoảng rất yếu

In [8]:
ID_COL = "SK_ID_CURR"
TARGET_COL = "TARGET"

n_bins = 20
min_bin_size = 0.03 # mỗi bin phải chứa ít nhất 3% điểm dữ liệu
numeric_unique = 15 # nếu một biến số có giá trị riêng biệt <= 15 thì sẽ coi là biến phân loại
min_freq_pct = 0.01 # Gộp category có tần suất < 1%


def prepare_feature(series):
        temp = series.copy()
        if is_numeric_dtype(temp):
            temp = temp.replace([np.inf,-np.inf], np.nan)
        return temp

def create_fine_bins(series, n_bins=n_bins, numeric_unique_threshold=numeric_unique, min_freq_pct=min_freq_pct):
        if is_numeric_dtype(series):
            # Nếu số giá trị riêng biệt <= threshold, coi như categorical
            if series.nunique() <= numeric_unique_threshold:
                bins = series.fillna("missing").astype(str)
            else:
                # Fine classing với qcut
                try:
                    bins = pd.qcut(series, q=n_bins, duplicates="drop")
                    bins = bins.astype("object")
                    bins[series.isna()] = "missing"
                except Exception:
                    # Fallback: nếu qcut lỗi, xử lý như categorical
                    bins = series.fillna("missing").astype(str)
        else:
            # Xử lý biến category
            # Lấy tần suất của từng category
            freq = series.value_counts(normalize=True)
            
            # Các category có tần suất < min_freq_pct -> gộp vào "OTHER"
            other_cats = freq[freq < min_freq_pct].index.tolist()
            
            bins = series.fillna("missing").astype(str)
            bins = bins.replace(other_cats, "OTHER")
        
        return bins


def statistics(feature: pd.Series, target: pd.Series, bins: pd.Series):
        temp = pd.DataFrame({
            "feature":feature,
            "target": target,
            "bins": bins
        })
        
        table = (temp.groupby("bins").agg(
            population = ("target","count"),
            bad = ("target",  "sum")
        ).reset_index())
        
        table["good"] = table["population"] - table["bad"]
        table["bad_rate"] = table["bad"] / table['population']
        table["population_pct"] = table["population"] / table["population"].sum()

        total_bad = table["bad"].sum()
        total_good = table["good"].sum()
        
        table["pct_bad"] = table["bad"] / total_bad
        table["pct_good"] = table["good"] / total_good
        
        table["pct_bad"] = table["pct_bad"].replace(0, 0.0000001)
        table["pct_good"] = table["pct_good"].replace(0, 0.0000001)
        
        table["woe"] = np.log(table["pct_bad"] / table["pct_good"])
        
        table["iv_contribution"] = (table["pct_bad"] - table["pct_good"]) * table["woe"]
        
        return table

def numeric_summary(table: pd.DataFrame, feature: pd.Series, bins = pd.Series):
        if not is_numeric_dtype(feature):
            return table
        temp = pd.DataFrame({
            "value": feature,
            "bins": bins
        })
        
        summary = (temp.groupby("bins").agg(
            min = ("value", "min"),
            max=("value", "max"),
            mean=("value", "mean")
        ).reset_index())
        
        table = table.merge(summary, on="bins", how= "left")
        
        return table

def fine_classing(feature: pd.Series, target: pd.Series, n_bins: int = n_bins):
        feature = prepare_feature(feature)
        bins = create_fine_bins(feature, n_bins=n_bins)
        report = statistics(feature, target, bins)
        report = numeric_summary(report, feature, bins)
        
        # Tính tổng IV của biến này
        iv_total = report["iv_contribution"].sum()
        
        # Thêm cột cảnh báo 
        report["warning"] = ""
        report.loc[report["population"] < (report["population"].sum() * 0.01), "warning"] += "Small sample; "
        report.loc[report["bad"] == 0, "warning"] += "Zero bad; "
        report.loc[report["good"] == 0, "warning"] += "Zero good; "
        
        return report, iv_total


def run_fine_classing_pipeline(df: pd.DataFrame, target_col: str = TARGET_COL, id_col: str = ID_COL) -> pd.DataFrame:
        # Lấy danh sách các biến (loại trừ ID và Target)
        feature_cols = [col for col in df.columns if col not in [id_col, target_col]]
        
        results = []
        
        print(f"Bắt đầu Fine Classing cho {len(feature_cols)} biến...")
        print("=" * 60)
        
        for i, col in enumerate(feature_cols, 1):
            try:
                # Chạy fine classing cho từng biến
                report, iv = fine_classing(df[col], df[target_col])
                
                # Lưu thông tin tổng hợp
                results.append({
                    "feature": col,
                    "data_type": "numeric" if is_numeric_dtype(df[col]) else "categorical",
                    "n_unique": df[col].nunique(),
                    "n_missing": df[col].isna().sum(),
                    "missing_pct": df[col].isna().mean(),
                    "n_bins": len(report),
                    "iv": iv,
                    "iv_strength": (
                        "Very Strong" if iv > 0.5 else
                        "Strong" if iv > 0.3 else
                        "Medium" if iv > 0.1 else
                        "Weak" if iv > 0.02 else
                        "Very Weak (Drop)"
                    ),
                    "has_warning": (report["warning"] != "").any(),
                    "min_population": report["population"].min(),
                    "max_population": report["population"].max(),
                    "report": report
                })
                
                print(f" [{i}/{len(feature_cols)}] {col}: IV = {iv:.4f} ({results[-1]['iv_strength']})")
                
            except Exception as e:
                print(f" [{i}/{len(feature_cols)}] Lỗi với biến {col}: {str(e)[:50]}...")
                results.append({
                    "feature": col,
                    "data_type": "unknown",
                    "n_unique": np.nan,
                    "n_missing": np.nan,
                    "missing_pct": np.nan,
                    "n_bins": 0,
                    "iv": np.nan,
                    "iv_strength": "Error",
                    "has_warning": False,
                    "min_population": np.nan,
                    "max_population": np.nan,
                    "report": None
                })
        
        print("=" * 60)
        print("Hoàn thành Fine Classing!")
        
        summary_df = pd.DataFrame(results)
        
        # Sắp xếp theo IV giảm dần
        summary_df = summary_df.sort_values("iv", ascending=False).reset_index(drop=True)
        
        return summary_df

In [9]:
summary_df = run_fine_classing_pipeline(screened_train)

Bắt đầu Fine Classing cho 185 biến...
 [1/185] CODE_GENDER: IV = 0.0388 (Weak)
 [2/185] FLAG_OWN_CAR: IV = 0.0064 (Very Weak (Drop))
 [3/185] FLAG_OWN_REALTY: IV = 0.0003 (Very Weak (Drop))
 [4/185] CNT_CHILDREN: IV = 0.0070 (Very Weak (Drop))
 [5/185] AMT_INCOME_TOTAL: IV = 0.0134 (Very Weak (Drop))
 [6/185] AMT_CREDIT: IV = 0.0534 (Weak)
 [7/185] AMT_ANNUITY: IV = 0.0323 (Weak)
 [8/185] AMT_GOODS_PRICE: IV = 0.1028 (Medium)
 [9/185] NAME_TYPE_SUITE: IV = 0.0027 (Very Weak (Drop))
 [10/185] NAME_INCOME_TYPE: IV = 0.0547 (Weak)
 [11/185] NAME_EDUCATION_TYPE: IV = 0.0496 (Weak)
 [12/185] NAME_FAMILY_STATUS: IV = 0.0228 (Weak)
 [13/185] NAME_HOUSING_TYPE: IV = 0.0150 (Very Weak (Drop))
 [14/185] REGION_POPULATION_RELATIVE: IV = 0.0455 (Weak)
 [15/185] DAYS_BIRTH: IV = 0.0885 (Weak)
 [16/185] DAYS_EMPLOYED: IV = 0.1154 (Medium)
 [17/185] DAYS_REGISTRATION: IV = 0.0290 (Weak)
 [18/185] OWN_CAR_AGE: IV = 0.0244 (Weak)
 [19/185] FLAG_EMP_PHONE: IV = 0.0327 (Weak)
 [20/185] FLAG_WORK_PHONE: I

In [10]:
summary_df

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,211355,122,0.000567,21,0.625719,Very Strong,True,122,10758,bins population bad go...
1,ext_min,numeric,105963,122,0.000567,21,0.473198,Strong,True,122,10936,bins population bad ...
2,ext_max,numeric,95512,122,0.000567,21,0.459759,Strong,True,122,10994,bins population bad go...
3,EXT_SOURCE_3,numeric,804,42680,0.198275,21,0.336042,Strong,False,7954,42680,bins population bad goo...
4,EXT_SOURCE_2,numeric,102229,464,0.002156,21,0.320308,Strong,True,464,10743,bins population bad ...
...,...,...,...,...,...,...,...,...,...,...,...,...
180,WEEKDAY_APPR_PROCESS_START,categorical,7,0,0.000000,7,0.000839,Very Weak (Drop),False,11322,37826,bins population bad good bad_rat...
181,FLAG_OWN_REALTY,categorical,2,0,0.000000,2,0.000258,Very Weak (Drop),False,65845,149412,bins population bad good bad_rate p...
182,worst_pos_dpd_12m,numeric,646,70442,0.327246,2,0.000059,Very Weak (Drop),False,70442,144815,bins population bad good...
183,bad_ratio_12m,numeric,209,70442,0.327246,2,0.000059,Very Weak (Drop),False,70442,144815,bins population bad good b...


In [11]:
pass_feature_table = summary_df[summary_df["iv_strength"] != "Very Weak (Drop)"]
pass_feature_table

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,211355,122,0.000567,21,0.625719,Very Strong,True,122,10758,bins population bad go...
1,ext_min,numeric,105963,122,0.000567,21,0.473198,Strong,True,122,10936,bins population bad ...
2,ext_max,numeric,95512,122,0.000567,21,0.459759,Strong,True,122,10994,bins population bad go...
3,EXT_SOURCE_3,numeric,804,42680,0.198275,21,0.336042,Strong,False,7954,42680,bins population bad goo...
4,EXT_SOURCE_2,numeric,102229,464,0.002156,21,0.320308,Strong,True,464,10743,bins population bad ...
...,...,...,...,...,...,...,...,...,...,...,...,...
95,applications_12m,numeric,49,11456,0.053220,8,0.020887,Weak,False,6910,128353,bins population bad good ba...
96,bad_ratio_all,numeric,2168,12570,0.058395,5,0.020253,Weak,False,9585,172299,bins population bad good...
97,avg_day_late_12m,numeric,12744,63084,0.293064,21,0.020235,Weak,False,7390,63084,bins population bad goo...
98,late_count_7,numeric,59,11034,0.051260,6,0.020194,Weak,False,6017,169551,bins population bad good b...


## Correlation Filter

Mục tiêu của bước này để loại bỏ những biến có tương quan cao (tránh cho việc mô hình xây dựng về sau có hiện tượng đa cộng tuyến).  
Đồng thời cũng là bước để ta cố gắng giảm thiểu các biến xuống với mong muốn có thể kiểm soát đươc rõ hơn.  

Ta cũng xây dựng một ánh xạ để ứng với từng khách hàng có giá trị x thuộc bins Y ở feature A thì sẽ có điểm Woe thế nào tương ứng.  

Output sau cùng sẽ là:   
    - Bảng thống kê các chỉ số của từng biến ứng như ở bước profiling, nhưng sẽ giảm đáng kể só biến  
    - Một dataframe mới của Train nhưng được đưa về điểm WOE (woe_df)   



In [12]:
CORRELATION_THRESHOLD = 0.7

def build_woe(report: pd.DataFrame) -> dict:
    bins_str = report["bins"].astype(str)
    woe_map = dict(zip(bins_str, report["woe"]))
    return woe_map

def transform_to_woe(feature_series: pd.Series, woe_map) -> pd.Series:
    temp = feature_series.copy()
    
    if is_numeric_dtype(temp):  
        bins = create_fine_bins(temp)  
        # Chuyển bins thành string trước khi map
        bins_str = bins.astype(str)
        woe_values = bins_str.map(woe_map)
    else:
        temp_fill = temp.fillna("missing").astype(str)
        woe_values = temp_fill.map(woe_map)
    return woe_values

def build_matrix_woe(df, pass_feature_table: pd.DataFrame) -> pd.DataFrame:
    df_woe = pd.DataFrame(index=df.index)
    
    for index, row in pass_feature_table.iterrows():
        feature_name = row["feature"]
        report = row["report"]  
        
        woe_map = build_woe(report)
        woe_values = transform_to_woe(df[feature_name], woe_map)
        df_woe[feature_name] = woe_values
    return df_woe

def corr_filter_matrix(df_woe, pass_feature_table: pd.DataFrame, threshold=CORRELATION_THRESHOLD) -> dict:
    feature = pass_feature_table["feature"].tolist()  
    iv_dict = dict(zip(pass_feature_table["feature"], pass_feature_table["iv"]))
    
    corr_matrix = df_woe[feature].corr()
    corr_mask = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
    
    high_corr_pair = []
    for i in range(len(feature)):
        for j in range(i, len(feature)):
            if corr_mask[i, j]:
                corr_value = corr_matrix.iloc[i, j]
                if abs(corr_value) > threshold:
                    high_corr_pair.append({
                        "feature_1": feature[i],
                        "feature_2": feature[j],
                        "correlation": corr_value
                    })
    
    if len(high_corr_pair) == 0:
        return {
            "final_feature_table": pass_feature_table.copy().sort_values("iv", ascending=False).reset_index(drop=True),
            "table_decision": pd.DataFrame()
        }
    
    decision = []
    feature_to_remove = set()
    for pair in high_corr_pair:
        f1 = pair["feature_1"]
        f2 = pair["feature_2"]
        corr_pair = pair["correlation"]
        iv_1 = iv_dict.get(f1, 0)
        iv_2 = iv_dict.get(f2, 0)

        if iv_1 >= iv_2:
            keep, drop = f1, f2
        else:
            keep, drop = f2, f1
            
        feature_to_remove.add(drop)
        decision.append({
            "feature_1": f1,
            "feature_2": f2, 
            "corr": corr_pair,
            "iv_1": iv_1,
            "iv_2": iv_2,
            "keep": keep,
            "drop": drop  
        })
    
    decision_df = pd.DataFrame(decision)
    
    removed_feature = list(feature_to_remove)  
    select_feature = [f for f in feature if f not in removed_feature]
    
    final_feature_table = pass_feature_table[pass_feature_table["feature"].isin(select_feature)].copy()
    final_feature_table = final_feature_table.sort_values("iv", ascending=False).reset_index(drop=True)
    
    return {
        "final_feature_table": final_feature_table,
        "table_decision": decision_df
    }

def main_corr(df, pass_feature_table):
    df_woe = build_matrix_woe(df, pass_feature_table)
    result = corr_filter_matrix(df_woe, pass_feature_table, CORRELATION_THRESHOLD) 
    
    return df_woe, result["final_feature_table"], result["table_decision"]

In [13]:
df_woe, final_feature_table, table_decision = main_corr(screened_train, pass_feature_table)
df_woe.head()

c:\Users\shina\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


,ext_mean,ext_min,ext_max,EXT_SOURCE_3,EXT_SOURCE_2,EXT_SOURCE_1,employment_years,DAYS_EMPLOYED,credit_utilization,AMT_GOODS_PRICE,...,BASEMENTAREA_AVG,NONLIVINGAREA_MODE,BASEMENTAREA_MEDI,bad_months_all_x,HOUSETYPE_MODE,applications_12m,bad_ratio_all,avg_day_late_12m,late_count_7,BASEMENTAREA_MODE
0,1.432985,1.293202,1.284070,1.216640,0.487135,1.083085,0.347325,0.352307,NaN,0.345027,...,-0.133391,-0.122638,-0.145933,0.301771,-0.156807,-0.065779,-0.047725,-0.266487,-0.047121,-0.126874
1,-0.058249,0.100533,-0.109407,0.160170,-0.294774,0.178629,0.263144,0.264309,NaN,-0.382073,...,-0.170176,-0.122638,-0.159762,-0.080940,-0.156807,-0.065779,-0.047725,-0.032824,-0.047121,-0.225985
2,-0.866510,-0.734270,-0.844287,-0.919028,-0.104687,0.062084,0.403462,0.399254,NaN,-0.228883,...,0.103605,0.119091,0.103605,-0.080940,0.133392,-0.065779,-0.047725,-0.032824,-0.047121,0.103605
3,0.758206,-0.023247,0.934163,0.160170,0.414441,0.062084,-0.121264,-0.121369,NaN,-0.103059,...,0.103605,0.119091,0.103605,-0.080940,0.133392,-0.065779,-0.047725,0.161147,0.289291,0.103605
4,-0.206743,-0.124654,-0.109407,-0.573715,0.301778,0.062084,0.091230,0.093690,NaN,-0.336651,...,0.103605,0.119091,0.103605,-0.080940,0.133392,-0.065779,0.414947,-0.016883,-0.047121,0.103605


In [14]:
final_feature_table

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,211355,122,0.000567,21,0.625719,Very Strong,True,122,10758,bins population bad go...
1,EXT_SOURCE_3,numeric,804,42680,0.198275,21,0.336042,Strong,False,7954,42680,bins population bad goo...
2,EXT_SOURCE_1,numeric,83961,121373,0.563852,21,0.158602,Medium,False,4694,121373,bins population ...
3,employment_years,numeric,11769,38756,0.180045,21,0.115402,Medium,False,8778,38756,bins population bad good ...
4,credit_utilization,numeric,129677,31535,0.146499,17,0.104111,Medium,True,1179,53925,bins population bad good...
5,AMT_GOODS_PRICE,numeric,828,187,0.000869,19,0.102781,Medium,True,187,24355,bins population bad ...
6,active_ratio,numeric,277,30836,0.143252,13,0.091637,Weak,True,517,36898,bins population bad good ...
7,age_years,numeric,17377,0,0.000000,20,0.088523,Weak,False,10753,10777,bins population ...
8,OCCUPATION_TYPE,categorical,18,67480,0.313486,15,0.081753,Weak,True,1480,67480,bins population bad ...
9,credit_goods_ratio,numeric,2767,187,0.000869,15,0.081629,Weak,True,187,76024,bins population bad good b...


## Manual Review

Đầu vào của Logistics Regrssion yêu cầu tính tuyến tính, vậy nên ta sẽ sử dụng kiến thức nghiệp vụ để xác định hướng của mỗi feature tương ứng (đơn điệu tăng hay giảm)  
Qua đó xây dựng hàm để ứng với kiến thức nghiệp vụ của biến A, ta muốn biến đó đơn điệu tăng hay giảm để thuận tiện cho việc giải thích model về sau.  
Đối với các biến khó xác định thì đối với quan điểm của tôi sẽ là giữ nguyên cấu trúc, vì việc ép nó là tăng hay giảm sẽ gây ra sự mất cấu trúc, dẫn tới model học sai rủi ro thực sự. Tuy nhiên việc để rõ cấu trúc có thể khiến model gặp tình trạng overfit quá mức mà không rõ nhóm rủi ro. Nên các biến này có thể được cân nhắc merge khoa học hơn ở các bước tiếp theo


Output:
- Bảng thống kê từng biến nhưng với các bins được cập nhật  
- Bảng data chỉ số thống kê cho tất cả các biến   
- Biến đổi lại woe_df ứng với report từng biến đã được cập nhật
 


In [15]:
def merge_bins(report: pd.DataFrame, left_idx: int, right_idx: int, eps: float = 1e-6) -> pd.DataFrame:
    """
    Merge bins with support for both numeric and categorical features.
    """
    df = report.copy().reset_index(drop=True)
    
    # Convert interval bins to string if needed
    if pd.api.types.is_interval_dtype(df['bins']):
        df['bins'] = df['bins'].astype(str)
    
    # ===== KIỂM TRA: CÓ PHẢI CATEGORICAL KHÔNG? =====
    # Nếu bins không chứa dấu phẩy và không có dạng "(a, b]"
    bins_str = df['bins'].astype(str)
    is_categorical = not any(',' in str(b) and ('(' in str(b) or '[' in str(b)) for b in bins_str)
    
    # Lưu lại để dùng
    left_bin = str(df.loc[left_idx, "bins"])
    right_bin = str(df.loc[right_idx, "bins"])
    
    # Gộp dữ liệu
    df.loc[left_idx, "population"] += df.loc[right_idx, "population"]
    df.loc[left_idx, "bad"] += df.loc[right_idx, "bad"]
    df.loc[left_idx, "good"] += df.loc[right_idx, "good"]
    
    # Xử lý min/max
    left_min = df.loc[left_idx, "min"]
    right_min = df.loc[right_idx, "min"]
    df.loc[left_idx, "min"] = left_min if pd.isna(right_min) or left_min <= right_min else right_min
    
    left_max = df.loc[left_idx, "max"]
    right_max = df.loc[right_idx, "max"]
    df.loc[left_idx, "max"] = left_max if pd.isna(right_max) or left_max >= right_max else right_max
    
    #  TẠO BINS MỚI 
    if is_categorical:
        # ===== CATEGORICAL: tạo bins dạng "1+2" hoặc "A+B" =====
        # Kiểm tra nếu là số
        if left_bin.replace('.', '').replace('-', '').isdigit() and right_bin.replace('.', '').replace('-', '').isdigit():
            df.loc[left_idx, "bins"] = f"{left_bin}+{right_bin}"
        else:
            # Nếu là text, gộp thành "A+B"
            # Nếu left_bin và right_bin đã có dấu +, thì không cần thêm
            if '+' in left_bin:
                # Nếu left_bin đã có +, chỉ cần thêm right_bin
                df.loc[left_idx, "bins"] = f"{left_bin}+{right_bin}"
            else:
                df.loc[left_idx, "bins"] = f"{left_bin}+{right_bin}"
    else:
        # ===== NUMERIC: tạo interval =====
        min_val = df.loc[left_idx, "min"]
        max_val = df.loc[right_idx, "max"]
        df.loc[left_idx, "bins"] = f"({min_val}, {max_val}]"
    
    # Drop right bins
    df = df.drop(index=right_idx).reset_index(drop=True)
    
    # Statistics
    total_population = df["population"].sum()
    total_good = df["good"].sum()
    total_bad = df["bad"].sum()
    
    df["population_pct"] = df["population"] / total_population if total_population > 0 else 0
    df["bad_rate"] = np.where(df["population"] > 0, df["bad"] / df["population"], 0)
    df["pct_good"] = df["good"] / total_good if total_good > 0 else 0
    df["pct_bad"] = df["bad"] / total_bad if total_bad > 0 else 0
    
    df["woe"] = np.log((df["pct_bad"] + eps) / (df["pct_good"] + eps))
    df["iv_contribution"] = (df["pct_bad"] - df["pct_good"]) * df["woe"]
    df["mean"] = df["bad_rate"]
    
    # Warnings
    warnings = []
    for _, row in df.iterrows():
        message = []
        if row["population_pct"] < 0.03:
            message.append("Small Bin")
        if row["bad"] == 0:
            message.append("Zero Bad")
        if row["good"] == 0:
            message.append("Zero Good")
        warnings.append(", ".join(message))
    
    df["warning"] = warnings
        
    return df

    
def find_breakpoint(report, trend):
    bad_rate = report["bad_rate"].values
    break_points = []
    
    if trend == "increasing":
        for i in range(len(bad_rate) -1):
            if bad_rate[i] > bad_rate[i+1]:
                break_points.append(i)
    elif trend == "decreasing":
        for i in range(len(bad_rate) - 1):
            if bad_rate[i] < bad_rate[i+1]:
                break_points.append(i)
    else:
        raise ValueError("trend have to be inc or dcr")
    
    return break_points

    

def rank_breakpoints(report, break_points: list):
    if len(break_points) ==0:
        return None

    bad_rate = report["bad_rate"].to_numpy()
    
    n = len(report)
    
    candidates = []
    
    for bp in break_points:
        if str(report.loc[bp,"bins"]).lower() == "missing":
            continue
        if (bp +1 <n) and (str(report.loc[bp + 1,"bins"]).lower() == "missing"):
            continue
        
        if bp - 1 >= 0 and str(report.loc[bp - 1, "bins"]).lower() == "missing":
            continue
        
        
        if bp ==0:
            score = abs(bad_rate[0] - bad_rate[1])
            candidates.append({
                "left":0,
                "right": 1,
                "score": score,
                "reason": "non_monotonic"
            })
            continue
        
        if bp == n-2:
            score = abs(bad_rate[n-2] - bad_rate[n-1])
            candidates.append({
                "left":n-2,
                "right": n-1,
                "score": score,
                "reason": "non_monotonic"
            })
            continue
        
        left_score = abs(bad_rate[bp] - bad_rate[bp - 1])
        right_score = abs(bad_rate[bp] - bad_rate[bp + 1])
        
        if left_score <= right_score:
            candidates.append({

                "left":bp-1,

                "right":bp,

                "score":left_score,

                "reason":"non_monotonic"

            })
        else:
            candidates.append({

                "left":bp,

                "right":bp+1,

                "score":right_score,

                "reason":"non_monotonic"

            })
    if not candidates:
        return None
    best = min(candidates, key = lambda x: x["score"])
    return best

def main_manual(report, trend, max_iter, verbose: bool = True):
    current_report = report.copy()
    
    iteration = 1
    while iteration <= max_iter:
        if len(current_report) < 5: # số bins tối thiểu nên là 5
            break
        break_points = find_breakpoint(current_report, trend= trend)
        if len(break_points) == 0:
            if verbose:
                print("=" * 60)
                print("Finished.")
                print("No breakpoint detected.")
                print(f"Total iterations : {iteration-1}")
                print("=" * 60)

            return current_report

        merge_info = rank_breakpoints(current_report, break_points)
        if merge_info is None:
            return current_report
        
        current_report = merge_bins(current_report, left_idx=merge_info["left"], right_idx= merge_info["right"])
        
        iteration += 1
    return current_report

### Phân nhóm biên

In [16]:
# nhóm increasing
INCREASING = [
    "late_count_7",
    "avg_day_late_12m",
    "credit_utilization",
    "credit_goods_ratio",
    "active_ratio",
    "late_payment_rate",
    "avg_credit_application_ratio",
    "active_loans",
    "REGION_RATING_CLIENT_W_CITY",
    "total_debt",
    "avg_drawings",
    "refused_count",
    "util_trend",
    "AMT_ANNUITY",
    "avg_day_late",
    "overdue_loans",
    "latest_annuity",
    "bad_months_12m_x",
    "bad_ratio_all",
    "applications_12m",
]

# nhóm decreasing
DECREASING = [
    "age_years",
    "employment_years",
    "ext_mean",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "credit_history_years",
    "approval_rate",
    "payment_ratio_last",
    "avg_payment_12m",
    "closed_loans",
    "months_observed_all",
    "DAYS_LAST_PHONE_CHANGE",
    "DAYS_REGISTRATION"
]

REVIEW_NUMERIC = [
    "AMT_CREDIT",
    "AMT_GOODS_PRICE",
    "total_credit",
    "avg_application_amount",
    "latest_enddate",
    "mean_future_instalments",
    "OWN_CAR_AGE",
    "ext_std",
    "FLOORSMAX_MEDI",
]

CATEGORICAL = [
    "ORGANIZATION_TYPE",
    "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE",
    "CODE_GENDER",
    "FLAG_DOCUMENT_3",
    "NAME_FAMILY_STATUS",
    "REG_CITY_NOT_WORK_CITY",
]

features_to_drop = ["avg_drawings","util_trend", "OWN_CAR_AGE", 
                    "payment_ratio_last", "avg_util_all"]

In [17]:
def summary_new_report(report):

    iv = report["iv_contribution"].sum()

    return {
        "n_bins": len(report),
        "iv": iv,
        "iv_strength": (
            "Very Strong" if iv > 0.5 else
            "Strong" if iv > 0.3 else
            "Medium" if iv > 0.1 else
            "Weak" if iv > 0.02 else
            "Very Weak (Drop)"
        ),
        "has_warning": (report["warning"] != "").any(),
        "min_population": report["population"].min(),
        "max_population": report["population"].max(),
    }


def update_manual_report(final_feature_table, manual_feature, trend,max_iter, verbose=False,):
    """
    Update report and feature summary after manual coarse classing.
    """

    result = final_feature_table.copy()

    manual_set = set(manual_feature)

    for idx, row in result.iterrows():

        feature = row["feature"]

        if feature not in manual_set:
            continue

        report = row["report"]

        # Merge bins
        new_report = main_manual(
            report=report,
            trend=trend,
            max_iter=max_iter,
            verbose=verbose,
        )

        summary = summary_new_report(new_report)

        # Update report
        result.at[idx, "report"] = new_report

        # Update data
        for col, value in summary.items():
            result.at[idx, col] = value
    
    return result

result = update_manual_report(final_feature_table, INCREASING, "increasing",10)
result_final_feature_table = update_manual_report(result, DECREASING, "decreasing", 10)

result_final_feature_table
    

C:\Users\shina\AppData\Local\Temp\ipykernel_17024\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval_dtype(df['bins']):
C:\Users\shina\AppData\Local\Temp\ipykernel_17024\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval_dtype(df['bins']):
C:\Users\shina\AppData\Local\Temp\ipykernel_17024\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval_dtype(df['bins']):
C:\Users\shina\AppData\Local\Temp\ipykernel_17024\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,211355,122,0.000567,20,0.625704,Very Strong,True,122,21513,bins ...
1,EXT_SOURCE_3,numeric,804,42680,0.198275,18,0.335636,Strong,False,7954,42680,bins ...
2,EXT_SOURCE_1,numeric,83961,121373,0.563852,18,0.158508,Medium,True,4694,121373,bins ...
3,employment_years,numeric,11769,38756,0.180045,14,0.114913,Medium,False,8816,38756,bins ...
4,credit_utilization,numeric,129677,31535,0.146499,15,0.103980,Medium,True,1179,53925,bins ...
5,AMT_GOODS_PRICE,numeric,828,187,0.000869,19,0.102781,Medium,True,187,24355,bins population bad ...
6,active_ratio,numeric,277,30836,0.143252,8,0.091330,Weak,False,13145,51074,bins population bad good ...
7,age_years,numeric,17377,0,0.000000,16,0.088503,Weak,False,10754,32295,bins p...
8,OCCUPATION_TYPE,categorical,18,67480,0.313486,15,0.081753,Weak,True,1480,67480,bins population bad ...
9,credit_goods_ratio,numeric,2767,187,0.000869,9,0.073611,Weak,True,187,86209,bins p...


In [18]:
result_final_feature_table[result_final_feature_table["feature"] == "ext_mean"]["report"].iloc[0]

,bins,population,bad,good,bad_rate,population_pct,pct_bad,pct_good,woe,iv_contribution,min,max,mean,warning
0,"(-0.0009885, 0.235]",10757,2894,7863,0.269034,0.049973,0.166542,0.039736,1.432966,0.181708,0.000011,0.235421,0.269034,
1,"(0.235, 0.304]",10757,2053,8704,0.190852,0.049973,0.118145,0.043986,0.988018,0.073270,0.235432,0.303743,0.190852,
2,"(0.304, 0.35]",10757,1698,9059,0.157851,0.049973,0.097715,0.045780,0.758194,0.039377,0.303750,0.349520,0.157851,
3,"(0.35, 0.384]",10756,1408,9348,0.130904,0.049968,0.081027,0.047241,0.539512,0.018228,0.349522,0.384212,0.130904,
4,"(0.384, 0.414]",10758,1222,9536,0.113590,0.049977,0.070323,0.048191,0.377922,0.008364,0.384214,0.413572,0.113590,
5,"(0.414, 0.439]",10756,1035,9721,0.096225,0.049968,0.059561,0.049126,0.192623,0.002010,0.413573,0.439265,0.096225,
6,"(0.439, 0.462]",10756,909,9847,0.084511,0.049968,0.052311,0.049762,0.049935,0.000127,0.439265,0.462175,0.084511,
7,"(0.462, 0.484]",10757,823,9934,0.076508,0.049973,0.047361,0.050202,-0.058248,0.000165,0.462177,0.483731,0.076508,
8,"(0.484, 0.504]",10757,717,10040,0.066654,0.049973,0.041261,0.050738,-0.206739,0.001959,0.483733,0.504429,0.066654,
9,"(0.504, 0.524]",10757,677,10080,0.062936,0.049973,0.038960,0.050940,-0.268118,0.003212,0.504429,0.524272,0.062936,


In [19]:

def build_woe_dict(report: pd.DataFrame) -> dict:
    """Tạo dict mapping từ bins -> woe"""
    woe_dict = dict(zip(report["bins"].astype(str), report["woe"]))
    
    if "other" not in woe_dict:
        woe_dict["other"] = 0.0
    
    return woe_dict

def is_numeric_interval(interval_str: str) -> bool:
    """
    Kiểm tra xem interval có phải là numeric interval không.
    Numeric interval có dấu phẩy và dấu ngoặc: "(1, 2]"
    Categorical có dấu +: "1+2" hoặc không có dấu phẩy
    """
    if not interval_str:
        return False
    # Nếu có dấu phẩy và có dấu ngoặc → numeric interval
    if ',' in interval_str and ('(' in interval_str or '[' in interval_str):
        return True
    return False



def parse_interval(interval_str):
    """
    Parse interval string thành (left, right)
    Chỉ áp dụng cho numeric intervals
    """
    if interval_str.lower() in ['missing', 'other']:
        return None, None
    
    # Xóa các ký tự đặc biệt
    clean_str = interval_str.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
    parts = clean_str.split(",")
    
    if len(parts) != 2:
        return None, None
    
    left_str = parts[0].strip()
    right_str = parts[1].strip()
    
    # Xử lý -inf và inf
    if left_str.lower() in ["-inf", "-infinity"]:
        left = float("-inf")
    elif left_str.lower() in ["inf", "infinity"]:
        left = float("inf")
    else:
        numbers = re.findall(r"-?\d+\.?\d*", left_str)
        if not numbers:
            return None, None
        left = float(numbers[0])
    
    if right_str.lower() in ["-inf", "-infinity"]:
        right = float("-inf")
    elif right_str.lower() in ["inf", "infinity"]:
        right = float("inf")
    else:
        numbers = re.findall(r"-?\d+\.?\d*", right_str)
        if not numbers:
            return None, None
        right = float(numbers[0])
    
    return left, right

def assign_bins(feature: pd.Series, report: pd.DataFrame) -> pd.Series:
    """
    Gán bin cho feature, hỗ trợ cả numeric và categorical
    """
    bins = pd.Series(index=feature.index, dtype="object")
    
    # Xử lý missing
    bins[feature.isna()] = "missing"
    
    numeric_report = report[report["bins"] != "missing"]
    
    for _, row in numeric_report.iterrows():
        interval = str(row["bins"])
        
        # ===== XÁC ĐỊNH LOẠI BIN =====
        if is_numeric_interval(interval):
            # ===== NUMERIC INTERVAL =====
            left, right = parse_interval(interval)
            
            if left is None or right is None:
                continue
            
            # Tạo mask dựa trên left và right
            if left == float("-inf"):
                mask = (feature.notna() & (feature <= right))
            elif right == float("inf"):
                mask = (feature.notna() & (feature > left))
            else:
                mask = (feature.notna() & (feature > left) & (feature <= right))
            
            bins.loc[mask] = interval
            
        else:
            # ===== CATEGORICAL =====
            # Nếu có dấu + → gộp nhiều categories
            if '+' in interval:
                categories = [cat.strip() for cat in interval.split('+')]
                mask = (feature.notna() & (feature.astype(str).isin(categories)))
                bins.loc[mask] = interval
            else:
                # Single category
                mask = (feature.notna() & (feature.astype(str) == interval))
                bins.loc[mask] = interval
    
    # Gán "other" cho giá trị chưa được gán
    bins[bins.isna()] = "other"
    
    return bins


def transform_to_woe_manual(feature, report):
    """Chuyển đổi feature thành WOE values"""
    woe_dict = build_woe_dict(report)
    bins = assign_bins(feature, report)
    return bins.map(woe_dict)

def build_matrix_woe_manual(df, result_final_feature_table):
    """Xây dựng ma trận WOE"""
    df_woe = pd.DataFrame(index=df.index)
    
    for _, row in result_final_feature_table.iterrows():
        feature = row["feature"]
        report = row["report"]
        
        print(f"Processing feature: {feature}")
        df_woe[feature] = transform_to_woe_manual(df[feature], report)
    
    return df_woe

In [20]:
feature_result = result_final_feature_table["feature"].tolist()
last_df = screened_train.loc[:, feature_result]
df_woe_logistics = build_matrix_woe_manual(last_df, result_final_feature_table)
df_woe_logistics

Processing feature: ext_mean
Processing feature: EXT_SOURCE_3
Processing feature: EXT_SOURCE_1
Processing feature: employment_years
Processing feature: credit_utilization
Processing feature: AMT_GOODS_PRICE
Processing feature: active_ratio
Processing feature: age_years
Processing feature: OCCUPATION_TYPE
Processing feature: credit_goods_ratio
Processing feature: credit_history_years
Processing feature: avg_util_all
Processing feature: approval_rate
Processing feature: late_payment_rate
Processing feature: latest_enddate
Processing feature: ORGANIZATION_TYPE
Processing feature: payment_ratio_last
Processing feature: avg_credit_application_ratio
Processing feature: active_loans
Processing feature: AMT_CREDIT
Processing feature: total_debt
Processing feature: REGION_RATING_CLIENT_W_CITY
Processing feature: avg_drawings
Processing feature: NAME_EDUCATION_TYPE
Processing feature: refused_count
Processing feature: DAYS_LAST_PHONE_CHANGE
Processing feature: avg_payment_12m
Processing feature:

,ext_mean,EXT_SOURCE_3,EXT_SOURCE_1,employment_years,credit_utilization,AMT_GOODS_PRICE,active_ratio,age_years,OCCUPATION_TYPE,credit_goods_ratio,...,latest_annuity,mean_future_instalments,OWN_CAR_AGE,bad_months_12m_x,NAME_FAMILY_STATUS,total_credit,applications_12m,bad_ratio_all,avg_day_late_12m,late_count_7
0,1.432966,1.216619,1.083050,0.347318,-0.141499,0.345027,-0.368221,0.390646,0.297606,0.091151,...,-0.014827,0.052389,0.055540,-0.073872,0.230885,-0.131852,-0.065779,-0.047725,-0.192917,-0.047121
1,-0.058248,0.160170,0.178621,0.261766,-0.396758,-0.382073,-0.368221,-0.067393,-0.239979,0.000000,...,-0.412427,-0.236809,0.055540,-0.073872,-0.074798,-0.040703,-0.065779,-0.047725,-0.032824,-0.047121
2,-0.866483,-0.892602,0.062084,0.380559,-0.396758,-0.228883,0.000000,-0.199889,0.297606,-0.226147,...,0.055133,0.052140,0.196768,-0.073872,0.230885,-0.045248,-0.065779,-0.047725,-0.032824,-0.047121
3,0.758194,0.160170,0.062084,-0.138001,-0.396758,-0.103059,0.000000,-0.349789,-0.239979,-0.226147,...,-0.014827,-0.057800,0.055540,-0.073872,0.230885,-0.028666,-0.065779,-0.047725,0.164660,0.000000
4,-0.206739,-0.573697,0.062084,0.099808,0.133992,-0.336651,-0.243636,-0.067393,0.297606,-0.226147,...,-0.014827,0.052140,0.055540,-0.073872,-0.074798,-0.102800,-0.065779,0.414947,-0.045358,-0.047121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215252,-0.346308,0.160170,0.062084,-0.561299,0.259022,0.088992,0.264632,-0.067393,0.184144,0.724179,...,-0.412427,-0.236809,0.055540,0.264633,-0.008187,0.264633,-0.065779,0.321206,0.065771,0.276201
215253,1.432966,0.160170,0.062084,-0.429644,0.259022,0.088992,0.264632,-0.429488,-0.233207,0.103724,...,-0.014827,0.052140,0.055540,0.264633,-0.333768,0.264633,-0.065779,-0.047725,-0.032824,-0.047121
215254,-0.206739,0.822374,-0.927939,-0.724935,0.133992,-0.108693,0.013365,-0.006907,-0.292244,0.091151,...,0.055133,0.052140,0.055540,-0.073872,-0.008187,-0.086966,-0.065779,0.321206,-0.032824,-0.047121
215255,-0.711590,-0.699244,0.062084,-0.389936,-0.396758,0.345027,0.000000,0.213758,0.297606,0.091151,...,-0.014827,0.052389,0.055540,-0.073872,-0.074798,0.226301,0.021074,-0.047725,-0.192917,-0.047121


In [21]:
# loại những biến sau vì tỉ lệ missing quá cao, và k có ý nghĩa
# Danh sách features cần loại bỏ


# Loại bỏ khỏi WOE matrix
cols_drop = features_to_drop + ["overdue_loans"]
df_woe_logistics_clean = df_woe_logistics.drop(columns=cols_drop)

print(f"Original shape: {df_woe_logistics.shape}")
print(f"After dropping: {df_woe_logistics_clean.shape}")
print(f"Removed {len(features_to_drop)} features")

Original shape: (215257, 50)
After dropping: (215257, 44)
Removed 5 features


In [22]:
df_woe_new = df_woe_logistics_clean.copy()
df_woe_new['SK_ID_CURR'] = train['SK_ID_CURR'].values
df_woe_new['TARGET'] = train['TARGET'].values

df_woe_new

,ext_mean,EXT_SOURCE_3,EXT_SOURCE_1,employment_years,credit_utilization,AMT_GOODS_PRICE,active_ratio,age_years,OCCUPATION_TYPE,credit_goods_ratio,...,mean_future_instalments,bad_months_12m_x,NAME_FAMILY_STATUS,total_credit,applications_12m,bad_ratio_all,avg_day_late_12m,late_count_7,SK_ID_CURR,TARGET
0,1.432966,1.216619,1.083050,0.347318,-0.141499,0.345027,-0.368221,0.390646,0.297606,0.091151,...,0.052389,-0.073872,0.230885,-0.131852,-0.065779,-0.047725,-0.192917,-0.047121,100002,1
1,-0.058248,0.160170,0.178621,0.261766,-0.396758,-0.382073,-0.368221,-0.067393,-0.239979,0.000000,...,-0.236809,-0.073872,-0.074798,-0.040703,-0.065779,-0.047725,-0.032824,-0.047121,100003,0
2,-0.866483,-0.892602,0.062084,0.380559,-0.396758,-0.228883,0.000000,-0.199889,0.297606,-0.226147,...,0.052140,-0.073872,0.230885,-0.045248,-0.065779,-0.047725,-0.032824,-0.047121,100004,0
3,0.758194,0.160170,0.062084,-0.138001,-0.396758,-0.103059,0.000000,-0.349789,-0.239979,-0.226147,...,-0.057800,-0.073872,0.230885,-0.028666,-0.065779,-0.047725,0.164660,0.000000,100007,0
4,-0.206739,-0.573697,0.062084,0.099808,0.133992,-0.336651,-0.243636,-0.067393,0.297606,-0.226147,...,0.052140,-0.073872,-0.074798,-0.102800,-0.065779,0.414947,-0.045358,-0.047121,100008,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215252,-0.346308,0.160170,0.062084,-0.561299,0.259022,0.088992,0.264632,-0.067393,0.184144,0.724179,...,-0.236809,0.264633,-0.008187,0.264633,-0.065779,0.321206,0.065771,0.276201,456248,0
215253,1.432966,0.160170,0.062084,-0.429644,0.259022,0.088992,0.264632,-0.429488,-0.233207,0.103724,...,0.052140,0.264633,-0.333768,0.264633,-0.065779,-0.047725,-0.032824,-0.047121,456252,0
215254,-0.206739,0.822374,-0.927939,-0.724935,0.133992,-0.108693,0.013365,-0.006907,-0.292244,0.091151,...,0.052140,-0.073872,-0.008187,-0.086966,-0.065779,0.321206,-0.032824,-0.047121,456253,0
215255,-0.711590,-0.699244,0.062084,-0.389936,-0.396758,0.345027,0.000000,0.213758,0.297606,0.091151,...,0.052389,-0.073872,-0.074798,0.226301,0.021074,-0.047725,-0.192917,-0.047121,456254,1


In [23]:
result_final_feature_table_last = result_final_feature_table[~result_final_feature_table["feature"].isin(cols_drop)]

result_final_feature_table_last

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,211355,122,0.000567,20,0.625704,Very Strong,True,122,21513,bins ...
1,EXT_SOURCE_3,numeric,804,42680,0.198275,18,0.335636,Strong,False,7954,42680,bins ...
2,EXT_SOURCE_1,numeric,83961,121373,0.563852,18,0.158508,Medium,True,4694,121373,bins ...
3,employment_years,numeric,11769,38756,0.180045,14,0.114913,Medium,False,8816,38756,bins ...
4,credit_utilization,numeric,129677,31535,0.146499,15,0.103980,Medium,True,1179,53925,bins ...
5,AMT_GOODS_PRICE,numeric,828,187,0.000869,19,0.102781,Medium,True,187,24355,bins population bad ...
6,active_ratio,numeric,277,30836,0.143252,8,0.091330,Weak,False,13145,51074,bins population bad good ...
7,age_years,numeric,17377,0,0.000000,16,0.088503,Weak,False,10754,32295,bins p...
8,OCCUPATION_TYPE,categorical,18,67480,0.313486,15,0.081753,Weak,True,1480,67480,bins population bad ...
9,credit_goods_ratio,numeric,2767,187,0.000869,9,0.073611,Weak,True,187,86209,bins p...


In [28]:
cols = result_final_feature_table_last["feature"].tolist()

In [25]:
#df_woe_new.to_csv("woe_logistics.csv", index=False, encoding="utf-8-sig")

In [26]:
def extract_woe_bins(result_final_feature_table: pd.DataFrame) -> pd.DataFrame:
    table = []
    
    for feature_name, row in result_final_feature_table.iterrows():
        report = row["report"]
        
        temp = report.copy()
        temp["feature"] = row["feature"]
        temp["bins"] = temp["bins"].astype(str)
        
        keep_cols = ["feature", "bins", "woe", "bad_rate", "population", 
                     "population_pct", "min", "max", "mean", "warning"]
        existing_cols = [col for col in keep_cols if col in temp.columns]
        temp = temp[existing_cols]
        
        table.append(temp)
    return pd.concat(table, ignore_index= True) if table else pd.DataFrame()


In [27]:
bins_feature = extract_woe_bins(result_final_feature_table_last)
bins_feature

#bins_feature.to_csv("bins_feature.csv", index=False,encoding="utf-8-sig")

,feature,bins,woe,bad_rate,population,population_pct,min,max,mean,warning
0,ext_mean,"(-0.0009885, 0.235]",1.432966,0.269034,10757,0.049973,0.000011,0.235421,0.269034,
1,ext_mean,"(0.235, 0.304]",0.988018,0.190852,10757,0.049973,0.235432,0.303743,0.190852,
2,ext_mean,"(0.304, 0.35]",0.758194,0.157851,10757,0.049973,0.303750,0.349520,0.157851,
3,ext_mean,"(0.35, 0.384]",0.539512,0.130904,10756,0.049968,0.349522,0.384212,0.130904,
4,ext_mean,"(0.384, 0.414]",0.377922,0.113590,10758,0.049977,0.384214,0.413572,0.113590,
...,...,...,...,...,...,...,...,...,...,...
479,avg_day_late_12m,missing,-0.032824,0.078324,63084,0.293064,NaN,NaN,0.078324,
480,late_count_7,"(-0.001, 1.0]",-0.047121,0.077298,169551,0.787668,0.000000,1.000000,0.077298,
481,late_count_7,"(2.0, 6.0]",0.276201,0.103743,25621,0.119025,2.000000,6.000000,0.103743,
482,late_count_7,"(6.0, 143.0]",0.285693,0.104629,9051,0.042047,7.000000,143.000000,0.104629,


Thực hiện map woe cho cả Validation và Test dataset

In [41]:
woe_validation_build = build_matrix_woe_manual(validation, result_final_feature_table_last)
woe_validation = woe_validation_build.copy()
woe_validation['SK_ID_CURR'] = validation['SK_ID_CURR'].values
woe_validation['TARGET'] = validation['TARGET'].values


woe_validation.to_csv("woe_logistics_validation.csv", index=False, encoding="utf-8-sig")
woe_validation

Processing feature: ext_mean
Processing feature: EXT_SOURCE_3
Processing feature: EXT_SOURCE_1
Processing feature: employment_years
Processing feature: credit_utilization
Processing feature: AMT_GOODS_PRICE
Processing feature: active_ratio
Processing feature: age_years
Processing feature: OCCUPATION_TYPE
Processing feature: credit_goods_ratio
Processing feature: credit_history_years
Processing feature: approval_rate
Processing feature: late_payment_rate
Processing feature: latest_enddate
Processing feature: ORGANIZATION_TYPE
Processing feature: avg_credit_application_ratio
Processing feature: active_loans
Processing feature: AMT_CREDIT
Processing feature: total_debt
Processing feature: REGION_RATING_CLIENT_W_CITY
Processing feature: NAME_EDUCATION_TYPE
Processing feature: refused_count
Processing feature: DAYS_LAST_PHONE_CHANGE
Processing feature: avg_payment_12m
Processing feature: CODE_GENDER
Processing feature: FLOORSMAX_MEDI
Processing feature: ext_std
Processing feature: closed_lo

,ext_mean,EXT_SOURCE_3,EXT_SOURCE_1,employment_years,credit_utilization,AMT_GOODS_PRICE,active_ratio,age_years,OCCUPATION_TYPE,credit_goods_ratio,...,mean_future_instalments,bad_months_12m_x,NAME_FAMILY_STATUS,total_credit,applications_12m,bad_ratio_all,avg_day_late_12m,late_count_7,SK_ID_CURR,TARGET
0,-0.968557,0.160170,0.062084,-0.138001,0.259022,0.169192,0.264632,-0.199889,0.297606,-0.226147,...,-0.279817,0.264633,0.228661,0.264633,0.261026,-0.047725,0.065771,-0.047121,100006,0
1,-0.968557,-0.197889,-0.927939,-0.138001,-0.141499,-0.681640,-0.368221,0.165568,-0.608646,-0.195892,...,-0.480461,-0.073872,-0.074798,-0.082257,0.092537,-0.047725,-0.045358,-0.047121,100009,0
2,-1.160659,0.160170,-0.927939,0.380559,0.259022,-0.762624,0.264632,-0.006907,0.297606,-0.134770,...,-0.215096,0.264633,-0.074798,0.264633,0.021074,-0.047725,-0.192917,-0.047121,100018,0
3,-0.346308,-0.197889,0.062084,-0.008172,-0.321023,-0.336651,-0.243636,0.313908,-0.239979,0.103724,...,0.052140,-0.073872,0.230885,-0.068242,0.021074,-0.047725,-0.192917,-0.047121,100023,0
4,0.758194,0.160170,-0.336667,-0.291122,0.259022,0.556322,0.264632,-0.136343,0.297606,-0.226147,...,-0.200432,0.264633,-0.074798,0.264633,-0.316487,-0.202715,-0.032824,-0.312669,100024,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46122,0.758194,0.232914,-0.101639,-0.008172,-0.396758,0.556322,-0.075725,0.165568,0.184144,0.091151,...,0.052140,-0.073872,-0.074798,-0.040703,-0.065779,-0.047725,-0.016704,-0.047121,456219,0
46123,0.192623,0.822374,0.062084,0.261766,-0.240756,-0.041488,-0.243636,-0.023685,-0.233207,0.394118,...,-0.215096,-0.073872,0.228661,-0.069674,-0.065779,-0.047725,-0.192917,-0.047121,456230,0
46124,-0.811080,-0.867206,-0.310126,-0.008172,0.496512,-0.158740,-0.243636,0.355782,0.354446,0.103724,...,-0.057800,-0.073872,-0.074798,-0.069674,0.092537,-0.047725,-0.016704,-0.047121,456235,0
46125,0.049935,-0.513938,0.490960,0.380559,0.133992,-0.041488,-0.243636,0.213758,-0.239979,0.394118,...,0.052140,-0.073872,0.230885,-0.082257,-0.065779,-0.047725,0.264918,-0.047121,456247,0


In [ ]:
woe_test_buld = build_matrix_woe_manual(test, result_final_feature_table_last)
woe_test = woe_test_buld.copy()
woe_test["SK_ID_CURR"] = test["SK_ID_CURR"].values
woe_test["TARGET"] = test["TARGET"].values

#woe_test.to_csv("woe_logistics_test.csv", index=False, encoding="utf-8-sig")

woe_test

Processing feature: ext_mean
Processing feature: EXT_SOURCE_3
Processing feature: EXT_SOURCE_1
Processing feature: employment_years
Processing feature: credit_utilization
Processing feature: AMT_GOODS_PRICE
Processing feature: active_ratio
Processing feature: age_years
Processing feature: OCCUPATION_TYPE
Processing feature: credit_goods_ratio
Processing feature: credit_history_years
Processing feature: approval_rate
Processing feature: late_payment_rate
Processing feature: latest_enddate
Processing feature: ORGANIZATION_TYPE
Processing feature: avg_credit_application_ratio
Processing feature: active_loans
Processing feature: AMT_CREDIT
Processing feature: total_debt
Processing feature: REGION_RATING_CLIENT_W_CITY
Processing feature: NAME_EDUCATION_TYPE
Processing feature: refused_count
Processing feature: DAYS_LAST_PHONE_CHANGE
Processing feature: avg_payment_12m
Processing feature: CODE_GENDER
Processing feature: FLOORSMAX_MEDI
Processing feature: ext_std
Processing feature: closed_lo

,ext_mean,EXT_SOURCE_3,EXT_SOURCE_1,employment_years,credit_utilization,AMT_GOODS_PRICE,active_ratio,age_years,OCCUPATION_TYPE,credit_goods_ratio,...,mean_future_instalments,bad_months_12m_x,NAME_FAMILY_STATUS,total_credit,applications_12m,bad_ratio_all,avg_day_late_12m,late_count_7,SK_ID_CURR,TARGET
0,-0.268118,-0.892602,-0.488504,-0.429644,-0.396758,-0.403109,0.000000,-0.349789,-0.233207,-0.195892,...,0.052140,-0.073872,-0.074798,-0.065150,-0.065779,0.414947,0.164660,0.276201,100011,0
1,0.049935,0.232914,0.178621,0.347318,-0.141499,0.122377,-0.368221,0.390646,-0.239979,-0.226147,...,-0.215096,-0.073872,-0.074798,0.034026,-0.065779,-0.047725,0.058988,-0.047121,100014,0
2,-0.866483,-0.699244,-0.927939,-0.429644,-0.396758,-0.228883,0.000000,-0.349789,-0.233207,-0.195892,...,0.052140,-0.073872,-0.074798,-0.065150,-0.065779,0.321206,-0.032824,-0.047121,100015,0
3,1.432966,1.216619,0.062084,0.237422,0.229949,0.556322,0.013365,0.207706,0.354446,0.394118,...,-0.279817,-0.073872,-0.074798,-0.045248,-0.065779,-0.047725,-0.016704,-0.047121,100020,0
4,-0.866483,-0.339450,0.062084,-0.724935,-0.240756,0.046943,0.378776,-0.136343,0.297606,-0.226147,...,0.318661,-0.073872,-0.333768,-0.040703,-0.065779,-0.047725,-0.192917,-0.047121,100022,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46122,-0.811080,-0.573697,-0.488504,-0.237884,-0.240756,0.556322,0.378776,0.165568,0.000000,0.394118,...,0.318661,-0.073872,-0.074798,-0.028666,0.261026,-0.047725,-0.192917,-0.047121,456211,0
46123,-0.058248,-0.791599,-0.336667,0.380559,-0.294265,0.122377,0.378776,-0.136343,0.184144,0.103724,...,-0.236809,-0.073872,-0.074798,-0.040703,-0.065779,-0.047725,-0.032824,-0.047121,456212,0
46124,0.988018,0.581692,0.062084,-0.429644,-0.240756,-0.403109,0.013365,-0.429488,-0.233207,0.091151,...,0.052140,-0.073872,-0.074798,-0.085394,0.421053,-0.047725,-0.192917,-0.047121,456231,0
46125,-1.160659,0.232914,-1.335344,-0.389936,-0.321023,-0.382073,-0.426915,-0.349789,-0.233207,0.091151,...,0.052389,-0.073872,-0.074798,-0.321447,0.261026,-0.047725,0.058988,0.276201,456244,0
